# momentum-buffer-update — worked example 2: Warm-start a momentum buffer from a saved state dict

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `momentum-buffer-update`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When resuming training from a checkpoint, the momentum buffer should be restored alongside the parameters, not reset to zero. Resetting the buffer would discard the accumulated gradient direction and cause a spike in the loss curve. This worked example shows how to save and restore buffer state so training continues smoothly.

## Worked solution

**Step 1 — run a few steps to build up non-zero buffers.** We apply the standard `b ← μ·b + g` recurrence for several steps. After this, the buffer contains the exponentially-weighted history of gradients.

**Step 2 — save the buffer state.** We clone the buffer tensor and store it. In a real optimizer this lives in `optimizer.state[param]['momentum_buffer']`.

**Step 3 — simulate resuming from checkpoint.** Create a brand-new buffer initialized to zero, then copy the saved state into it using `.copy_()`. This is the warm-start.

**Step 4 — continue training from the warm-started buffer.** The next step's effective gradient should be identical to what it would have been had we never paused, because the buffer's value is the same.

**Step 5 — compare vs a cold-start (buffer reset to zero).** The cold-start buffer's first step produces only the raw gradient, while the warm-start immediately uses the accumulated momentum.

In [ ]:
import torch

torch.manual_seed(0)

def run_steps(buf, grads, mu):
    """Apply b <- mu*b + g for each gradient in grads. Mutates buf in-place."""
    for g in grads:
        buf.copy_(mu * buf + g)
    return buf

# Phase 1: build up buffer over 4 steps with synthetic gradients.
mu = 0.9
torch.manual_seed(5)
grads_phase1 = [torch.randn(4) for _ in range(4)]

buf_original = torch.zeros(4)
run_steps(buf_original, grads_phase1, mu)
print("Buffer after phase 1:", buf_original)

# Save state (simulate checkpoint).
saved_buf = buf_original.clone()

# Phase 2: one more gradient arrives.
torch.manual_seed(7)
g_next = torch.randn(4)

# Warm-start path: restore saved buffer and apply next step.
buf_warm = torch.zeros(4)
buf_warm.copy_(saved_buf)  # restore from checkpoint
buf_warm.copy_(mu * buf_warm + g_next)

# Cold-start path: buffer reset to zero, apply next step.
buf_cold = torch.zeros(4)
buf_cold.copy_(mu * buf_cold + g_next)  # mu*0 + g = g

print("Warm-start next buffer:", buf_warm)
print("Cold-start next buffer:", buf_cold)
print("Difference (should be non-zero):", (buf_warm - buf_cold).abs().max().item())